# T05 · PHL fine-tuning of MACE-OFF23_medium: energies, forces **and** Hessians to ωB97M-D3(BJ)/def2-TZVPPD

**openQHA theory notebook — the version of 2026-09-22 (PHL verbatim).** It rests on the choices
behind the run: the training target is the Cartesian Hessian matrix, sampled by Hessian–vector
products (PHL's loss as published); basin frames only (the msRRHO result is the deliverable; the
extrapolation rows are reference); validation by four probes fixed per frame; the Replay as a
drawn file from SPICE's train split (round 1: a 30,000-frame draw in two arms at `config_weight`
1 / 10 — the S0 rows R0–R4 are history); the judge's gate rows on the Hessian matrix
itself (frequencies, entropy, NMS / MD / VIB are post-processing); and production = the two arms
`replay30k_w1` / `replay30k_w10` (the mace-docs multihead guidance; user, 2026-09-30),
$w_H$ = the balance the driver measures, the gate closed (every row reported, nothing decided).
The code those choices became (`phl`, `phl_loss`, `run`, `dataset`, `judge`, the fork's
commit C, `s0_spice_pt_draw.py`; `04_dataset` prints `N_TRAIN_HESSIAN` and the two production
draw commands — the S0 ladder fields stay as history, `05_train --hessian-weight balance` is
the default, `06_judge --thermo-tag` reads the engine's
msRRHO Records from their own tag). It replaces the version of the same day that still carried
T03's projected, mass-weighted, entropy-weighted loss variants as "diagnostics" beside the target
and a section comparing where each norm spends its gradient: **nothing in the training is
projected except the random probe**, and the frequency rows of the judge come from the standard
vibrational analysis of a trained matrix (mass-weighting + Eckart projection), an evaluation step.

T04 is the derivation underneath this notebook (what E/F leaves free, PHL's estimator, its cost).
This one states the goal the project has, in the form PHL gave it: **take MACE-OFF23_medium and,
with energies, forces and Hessian-vector products in one loss, bring E, F and H to the accuracy of
the level the model was trained at** — ωB97M-D3(BJ)/def2-TZVPPD — on the molecules we care about,
**at their minima**. Not a new level, not a model from scratch, not a surface potential. Every
claim is checked in the cell after it on

* `tests/data/methyloxirane_frames/`: 2-methyloxirane (QM9 000044), the basin and four classical
  298 K displaced frames, each with **E, F and the analytic Hessian at both levels at identical
  positions** — the only local frames where the base model can be measured against its own level
  on all three quantities at once; in the campaign such displaced frames are **held-out
  generator frames**: labelled E/F, never trained on, read by the judge;
* `tests/data/propanal_molecule/`: the ωB97M analytic Hessian and the stored MACE-OFF23_medium
  Hessian at the same basin;
* the mace fork checkout (`mace/`), `data/training_sets/mace-off23_spice_index.dat` and
  MACE-OFF23_medium itself, when they load (the cells say "skipped" otherwise).

Reading order: §1 the goal as three numbers the base model currently fails → §2 PHL's loss
verbatim — the training target — sampled, unbiased, exact with $3N$ probes → §3 **basin frames
only**: what a stationary-only Hessian set leaves free, what Rodriguez measured, and
$\rho_k$ as the judge's *reference* reading → §4 `w_H` measured, not quoted (`05_train`'s default;
no scan) → §5 the training mechanics as the code now does them: the Replay by its file,
`config_weight`, the duplication threshold 0, commit C, the S0 rows R0–R4 (history) with their
coverage and the round-1 standard that runs (a 30,000-frame draw in two arms, `config_weight`
1 / 10); validation on fixed probes; the two stages → §6 the judge table: gate rows and
reference rows → §7 the recipe end to end, the smoke run first, then the two production arms → §8
**T05 against T04**, and what changed.

Conventions as in T04: $N$ atoms, $x\in\mathbb R^{3N}$ (Å), $E$ eV, $g=\nabla E=-F$,
$H=\nabla^2E$ (eV Å$^{-2}$), $\Delta H=H_\theta-H_r$, $\theta$ the model, $r$ the reference.

In [ ]:
import sys, math, time, re
from pathlib import Path
import numpy as np
import torch
torch.set_default_dtype(torch.float64)

def _repo_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "openqha" / "__init__.py").is_file():
            return p
    raise RuntimeError("run from inside the openQHA checkout")

ROOT = _repo_root(); sys.path.insert(0, str(ROOT))
from ase.data import atomic_masses, atomic_numbers
from openqha.qm_interfaces import orca
from openqha.thermochem import hessian as hessian_mod, hessian_compare as hc
from openqha.data import frames as frames_mod
from openqha_hessian import phl, hvp as hvp_mod, smoke_fit, judge

LEVEL = "wb97m-d3bj_def2-tzvppd"; MLEVEL = "mace-off23_medium"
to_cm = lambda lam: np.sign(lam) * np.sqrt(np.abs(lam)) * hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU
masses_of = lambda atoms: np.array([atomic_masses[atomic_numbers[s]] for s in atoms.get_chemical_symbols()])
EV_TO_KCAL = 23.0605

# 2-methyloxirane: basin + 4 displaced frames, E/F/H at both levels, identical positions
FIXM = ROOT / "tests" / "data" / "methyloxirane_frames"
fr_t = frames_mod.read_frames(FIXM / f"basin.{MLEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{MLEVEL}.extxyz")
fr_r = frames_mod.read_frames(FIXM / f"basin.{LEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{LEVEL}.extxyz")
assert all(np.abs(a.positions - b.positions).max() < 1e-7 for a, b in zip(fr_t, fr_r))
N = len(fr_t[0]); n3 = 3 * N; mm = masses_of(fr_t[0]); nF = len(fr_t)
X = [a.positions.copy() for a in fr_r]
E_r = np.array([a.get_potential_energy() for a in fr_r]); E_t = np.array([a.get_potential_energy() for a in fr_t])
F_r = [a.get_forces() for a in fr_r]; F_t = [a.get_forces() for a in fr_t]
H_r = [np.asarray(a.info["hessian"], float).reshape(n3, n3) for a in fr_r]
H_t = [np.asarray(a.info["hessian"], float).reshape(n3, n3) for a in fr_t]
rms = np.array([a.info["rms_displacement_A"] for a in fr_t])

# propanal basin for the propanal rows
FIXP = ROOT / "tests" / "data" / "propanal_molecule"
parsed = orca.parse_hess(FIXP / "msrrho" / f"orca.{LEVEL}.basin00.hess")
sym_p = list(parsed["symbols"]); xp = np.asarray(parsed["positions_bohr"]) / orca.BOHR_PER_ANGSTROM
mp = np.array([atomic_masses[atomic_numbers[s]] for s in sym_p])
Hp_r = orca.hessian_to_ev_per_angstrom2(parsed["hessian_eh_bohr2"]); Hp_t = np.load(FIXP / "mace" / "basin00" / f"hessian_at_{LEVEL}.npy")
print(f"2-methyloxirane ({fr_t[0].info['smiles']}): N = {N}, {nF} frames, RMS displacement {np.round(rms, 3)} Å;  propanal: N = {len(sym_p)}")
print(f"reference level {LEVEL} (ORCA {fr_r[0].info.get('orca_version', '?')}), engine {MLEVEL}")

2-methyloxirane (C[C@H]1CO1): N = 10, 5 frames, RMS displacement [0.    0.058 0.096 0.127 0.059] Å;  propanal: N = 10
reference level wb97m-d3bj_def2-tzvppd (ORCA 6.0.1), engine mace-off23_medium


## 1. The goal as three numbers — where the base model stands against its own level

MACE-OFF23_medium was trained on 951,813 SPICE frames labelled at ωB97M-D3(BJ)/def2-TZVPPD
with **energies and forces only** (Kovács et al., JACS 2025; `mace-off-main/train_scripts`,
$w_E{:}w_F = 40{:}1000$, 190 epochs, Stage Two from epoch 115). Its Hessians were never
supervised. "Reaching the DFT level" therefore means three things at once, each a number that
can fail on held-out frames labelled at that same level:

| quantity | ruler (shipped paths only) | claim |
|---|---|---|
| $E$ | per-atom energy error against the Label | no worse than the base where the base is in distribution; better on our molecules |
| $F$ | force RMSE | same |
| $H$ | full Cartesian `get_hessian` against the Label: the target $\|\Delta H\|_F^2/(9N^2)$ itself (the gate row), the element MAE, and — by the standard vibrational analysis of both matrices — the frequency MAE over **all** modes, the low-mode MAE, the mode overlap (`hessian_compare`; reference rows) | better than the base on every row where the base is measurably wrong; no worse anywhere |
| forgetting | $E/F$ on a fixed draw of SPICE's test split | within 15 % of the base |

The cell prints the base model's three numbers on the five frames, from the stored engine
values (no model needed). These are the numbers the fine-tune has to move.

In [2]:
def efh_row(k):
    dE = (E_t[k] - E_r[k]) / N * 1e3                                  # meV / atom, signed
    fr_ = np.sqrt(np.mean((F_t[k] - F_r[k]) ** 2)) * 1e3             # meV / Å
    c = hc.compare_hessians(H_t[k], H_r[k], mm, X[k])
    return dE, fr_, c

print("frame  RMS(Å)   ΔE/atom(meV)   F RMSE(meV/Å)   H MAE(eV/Å²)   ‖ΔH‖²/9N²   freq MAE(cm⁻¹)  low-mode MAE   softening   cos v1")
rows_base = []
for k in range(nF):
    dE, frm, c = efh_row(k); rows_base.append(c)   # the frequency columns: hessian_compare's standard analysis (evaluation)
    print(f"  {k}    {rms[k]:.3f}    {dE:+8.2f}       {frm:8.1f}        {c['HESSIAN_MAE']:.4f}       {np.linalg.norm(H_t[k] - H_r[k]) ** 2 / (9 * N * N):.4f}      {c['FREQ_MAE_CM']:6.1f}          {c['FREQ_MAE_LOW_CM']:5.1f}        {c['SOFTENING_SLOPE']:.4f}     {c['EIGVEC1_COS_ECKART']:.3f}")
rel = [(E_t[k] - E_t[0]) - (E_r[k] - E_r[0]) for k in range(1, nF)]
print(f"\nrelative energies E_k - E_0, model minus reference: {np.round(np.array(rel) * EV_TO_KCAL, 2)} kcal/mol")
print("-> the base model shares the reference's energy zero (E0s) and misses the relative energies by tenths of a kcal/mol, the")
print("   forces by 20-30 meV/Å, the Hessian by ~0.07-0.08 eV/Å² per element, 0.02-0.03 on the target ‖ΔH‖²/(9N²) and 5-6 cm⁻¹")
print("   per mode; none of the three is systematically worse at the displaced frames on this molecule. (A low-mode MAE of 0.0 means the frame has no reference mode below 300 cm⁻¹: at a")
print("   non-stationary geometry the projected spectrum is not the basin's.) Three quantities, one level, one fine-tune.")

frame  RMS(Å)   ΔE/atom(meV)   F RMSE(meV/Å)   H MAE(eV/Å²)   ‖ΔH‖²/9N²   freq MAE(cm⁻¹)  low-mode MAE   softening   cos v1
  0    0.000       +0.31           26.7        0.0730       0.0287         4.8           12.7        0.9990     1.000
  1    0.058       +0.16           21.3        0.0783       0.0230         5.6            7.9        1.0011     1.000
  2    0.096       +0.21           25.8        0.0709       0.0173         6.3            0.0        0.9993     1.000
  3    0.127       -0.05           29.5        0.0779       0.0271         5.2            0.0        1.0018     0.999
  4    0.059       +0.80           30.2        0.0845       0.0265         6.4            0.0        0.9984     1.000

relative energies E_k - E_0, model minus reference: [-0.03 -0.03 -0.08  0.11] kcal/mol
-> the base model shares the reference's energy zero (E0s) and misses the relative energies by tenths of a kcal/mol, the
   forces by 20-30 meV/Å, the Hessian by ~0.07-0.08 eV/Å² per element, 0.02-0

## 2. PHL's loss verbatim — the training target

PHL's reference implementation (`PHL_training.ipynb`, ANI, from scratch) computes per structure and
per minibatch one Gaussian probe $v\sim\mathcal N(0,I_{3N})$ and
$$\mathcal L^{\rm PHL} = \frac{(E_\theta-E_r)^2}{\sqrt N} + 0.30\,\frac{\|F_\theta-F_r\|^2}{3N}
 + 0.09\,\frac{\|H_\theta v - H_r v\|^2}{9N^2},\qquad H_\theta v=-\nabla_x(F_\theta\!\cdot v)\ \text{(one extra backward)} .\tag{2.1}$$
Hutchinson's identity $\mathbb E_v\|Av\|^2=\|A\|_F^2$ makes the third term an unbiased estimate of
$\|H_\theta-H_r\|_F^2/(9N^2)$ — PHL eq. 6, the mean squared error of the Cartesian matrix as stored:
**PHL is the full-Hessian loss, sampled** (T04 §2).

**The training target is that matrix, as in Rodriguez, PHL, HIP and PFT; frequencies,
modes and the msRRHO entropy are evaluation quantities computed from it afterwards, never terms of
the loss.** So (2.1)'s third term, with Gaussian probes (PHL's Algorithm 1, the
default; Rademacher kept as the smaller-variance choice) and $k$ probes per structure, is the Hessian loss,
$$\mathcal L_H=\frac1{9N^2k}\sum_{j=1}^k\|(H_\theta-H_r)\,v_j\|^2,\qquad \mathbb E\,\mathcal L_H=\frac{\|\Delta H\|_F^2}{9N^2} ,\tag{2.2}$$
and that is all of it: the probe the model sees is the raw draw, the reference side is a matvec on
the stored Label, nothing is mass-weighted or projected (`phl.make_probes`, `phl_loss.hvp_error`;
T04 Algorithms 2–3). With the $3N$ unit vectors as probes (2.2) is $\|\Delta H\|_F^2/(9N^2)$ exactly
and $H_\theta e_j$ is column $j$ of `get_hessian` — the judge's tool. The cell measures, on the five
frames: (2.2) against the exact value for one Gaussian probe over 4000 draws, for the code's own
$k=4$ Rademacher draw over 200 draws with the standard error T04 eq. 2.3 predicts, and for the $3N$
unit probes.

In [3]:
rng = np.random.default_rng(0)
print("frame   ‖ΔH‖²/(9N²) exact   Hutchinson, 4000 Gaussian v   phl.make_probes k=4 rademacher: mean of 200 draws (s.d. of one draw)   s.d. predicted (T04 2.3)   3N unit probes")
for k in range(nF):
    dH = H_t[k] - H_r[k]
    full = np.linalg.norm(dH) ** 2 / (9 * N * N)
    V = rng.standard_normal((4000, n3))
    hutch = np.mean(np.sum((V @ dH.T) ** 2, axis=1)) / (9 * N * N)
    ests = []
    for _ in range(200):
        vt4, r4, i4 = phl.make_probes(H_r[k], mode="rademacher", k=4, rng=rng)
        ests.append(phl.estimator_from_products(np.asarray([H_t[k] @ v for v in vt4]), r4, i4))
    se_pred = np.sqrt(phl.estimator_variance(H_t[k], H_r[k], k=4)["rademacher"])
    vt_all, r_all, i_all = phl.make_probes(H_r[k], mode="cartesian")
    unit = phl.estimator_from_products(np.asarray([H_t[k] @ v for v in vt_all]), r_all, i_all)
    print(f"  {k}       {full:.4e}            {hutch:.4e} ({hutch/full-1:+.1%})              {np.mean(ests):.4e} ({np.std(ests):.1e})                          {se_pred:.1e}                {unit:.4e}")
print("-> the training term (2.2) is the Cartesian Frobenius norm, sampled: every column agrees with the exact value within its noise,")
print("   the spread of one k = 4 draw is what T04 eq. 2.3 predicts, and the 3N unit probes reproduce it to every digit (= get_hessian).")

frame   ‖ΔH‖²/(9N²) exact   Hutchinson, 4000 Gaussian v   phl.make_probes k=4 rademacher: mean of 200 draws (s.d. of one draw)   s.d. predicted (T04 2.3)   3N unit probes
  0       2.8686e-02            2.8910e-02 (+0.8%)              2.7769e-02 (1.3e-02)                          1.5e-02                2.8686e-02
  1       2.2978e-02            2.2600e-02 (-1.6%)              2.2761e-02 (8.8e-03)                          9.0e-03                2.2978e-02
  2       1.7280e-02            1.7647e-02 (+2.1%)              1.7786e-02 (5.8e-03)                          5.9e-03                1.7280e-02
  3       2.7093e-02            2.7738e-02 (+2.4%)              2.7119e-02 (1.2e-02)                          1.2e-02                2.7093e-02
  4       2.6476e-02            2.6571e-02 (+0.4%)              2.5752e-02 (8.9e-03)                          9.2e-03                2.6476e-02
-> the training term (2.2) is the Cartesian Frobenius norm, sampled: every column agrees with the exact value

## 3. Basin frames only: what a stationary-only Hessian set leaves free, and who reads it

In PHL, Rodriguez and PFT every *training* structure has a Hessian label. Our labels cost
~25× an E/F label (an ORCA analytic Hessian at ωB97M-D3(BJ)/def2-TZVPPD), and the campaign's
bill was 22 days of 12 nodes for every frame against ~4 for the basin frames alone.
Rodriguez 2025 trained on **stationary points only** (RTP: 35,087 minima and transition
states) and evaluated on IRC and normal-mode-sampled frames: Hessian training at stationary
points cut the off-stationary Hessian RMSE by 70–90 % and halved the force error, without an
off-stationary Hessian in the training set (tables 2–4). The deliverable here is the msRRHO
thermochemistry at the molecules' own minima, which reads the basin Hessian and nothing else.

**The rule.** `dataset.build(train_generators=("basin",))`: only basin frames
enter `train` and `valid`; every labelled frame of another generator — `displaced`, `merged`,
`saddle` — is a **held-out generator** frame, routed to `test` with `held_out_generator = yes`
in the index, read by the judge as *reference* rows (binned by RMS displacement; §6 — post-processing)
and never trained on. T04 §1 proved that E/F labels say nothing about the curvature at or between
samples (Proposition 1); what a basin-only Hessian fine-tune buys away from the minimum is
therefore *measured*, not promised. The fixture has Hessians at the displaced frames, so the
cell can show what the reference rows will read before any training: the reference Hessian's
own change over a 298 K frame, and the base model's target error there relative to the basin,
$$\rho_k=\frac{\|\Delta H(x_k)\|_F^2/(9N^2)}{\|\Delta H(x_0)\|_F^2/(9N^2)}\tag{3.1}$$
(T04 eq. 3.3; the judge's RMS bins read the same ratio, engine against base).
$\rho_k\approx1$ after training means the basin Hessians carried; $\rho_k\gg1$ is the PFT symptom
(curvature roughened between samples) and is reported, not gated — no second training stage is
triggered by it.

In [ ]:
c0 = np.linalg.norm(H_t[0] - H_r[0]) ** 2 / (9 * N * N)
print("frame  generator   RMS(Å)   bin      ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖   ‖ΔH‖²/(9N²)   ρ_k (3.1)   freq MAE all(cm⁻¹)   freq MAE low   who reads it")
for k in range(nF):
    dHr = np.linalg.norm(H_r[k] - H_r[0]) / np.linalg.norm(H_r[0])
    cart = np.linalg.norm(H_t[k] - H_r[k]) ** 2 / (9 * N * N)
    c = rows_base[k]
    gen = fr_t[k].info["generator"]
    print(f"  {k}    {gen:10s}  {rms[k]:.3f}   {judge.rms_bin(rms[k] if gen != 'basin' else 0.0):6s}         {dHr:.3f}                    {cart:.4f}        {cart/c0:5.2f}         {c['FREQ_MAE_CM']:5.1f}              {c['FREQ_MAE_LOW_CM']:5.1f}        {'train / valid (the loss)' if gen == 'basin' else 'test: a reference bin (the judge)'}")
print("-> over a 298 K frame the reference curvature itself moves by 15-22 %; the base model's target error at the displaced frames is")
print("   ρ_k times its basin error (here below 1: the base is not worse off the minimum on this molecule). The fine-tune sees row 0")
print("   only; rows 1-4 are held-out generator frames whose E/F (and, in this fixture, H) the judge reads in the '<0.08'")
print("   and '<0.15' Å bins -- Rodriguez's extrapolation readout, reported and never gated.")

frame  generator   RMS(Å)   bin      ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖   ‖ΔH‖²/(9N²)   ρ_k (3.1)   freq MAE all(cm⁻¹)   freq MAE low   who reads it
  0    basin       0.000   0              0.000                    0.0287         1.00           4.8               12.7        train / valid (the loss)
  1    displaced   0.058   <0.08          0.149                    0.0230         0.80           5.6                7.9        test: a reference bin (the judge)
  2    displaced   0.096   <0.15          0.152                    0.0173         0.60           6.3                0.0        test: a reference bin (the judge)
  3    displaced   0.127   <0.15          0.223                    0.0271         0.94           5.2                0.0        test: a reference bin (the judge)
  4    displaced   0.059   <0.08          0.164                    0.0265         0.92           6.4                0.0        test: a reference bin (the judge)
-> over a 298 K frame the reference curvature itself moves 

## 4. $w_H$ is measured, not quoted

PHL's $(1, 0.30, 0.09)$ balance ANI's energy in Hartree against its own force and HVP terms;
Rodriguez's and PHL's $w_H/w_F\approx0.25$–$0.30$ band is Cartesian and unweighted. None of it
transfers to a MACE that already fits E and F at its own level: the Hessian term's size depends on
the molecules and on the base model's state (T04 §5 item 3). The rule used here: set $w_H$ so that the Hessian term enters the epoch-0 gradient
with the same share as the force term on the **base** model,
$$w_H=\frac{w_F\,\mathcal L_F}{\mathcal L_H}\Big|_{\theta=\text{base}},\tag{4.1}$$
and **no scan**. This is what `05_train.py` does by itself:
`--hessian-weight balance` is the default, `run.hessian_weight_balance` calls
`smoke_fit.epoch_zero_balance` on the base model over the run's own train file with the
Cartesian target (the full matrix per Hessian frame, not the estimator), and the Record keeps
`HESSIAN_WEIGHT_RULE = balance` with `BALANCE_L_E / BALANCE_L_F / BALANCE_L_H` beside the
`HESSIAN_WEIGHT` that ran; a number on the command line is `HESSIAN_WEIGHT_RULE = given`. The
cell computes the three terms of mace's loss on the five frames from the stored values (per-atom
energy MSE, per-component force MSE, the full matrix (2.2)) and the resulting $w_H$, then writes
the same five frames as a train file and lets the driver's rule measure it on the live base model
— the two numbers must agree, because the stored MACE values are that model's.

In [ ]:
wE, wF = 1.0, 100.0
L_E = np.mean(((E_t - E_r) / N) ** 2)
L_F = np.mean([np.mean((F_t[k] - F_r[k]) ** 2) for k in range(nF)])
L_H_cart = np.mean([np.linalg.norm(H_t[k] - H_r[k]) ** 2 / (9 * N * N) for k in range(nF)])     # (2.2), the target, exact
print(f"base model on the {nF} frames:  L_E = {L_E:.3e} (eV/atom)²   L_F = {L_F:.3e} (eV/Å)²   w_F L_F = {wF*L_F:.3e}")
print(f"  L_H (2.2, the target) = {L_H_cart:.3e}  ->  w_H (4.1) = {wF*L_F/L_H_cart:8.3f}")
print(f"  PHL's w_H/w_F = 0.09/0.30 = 0.30 would put the Cartesian Hessian term at {0.30*wF*L_H_cart/(wF*L_F):.1f}x the force term here")
print("-> the number depends on the data set and on the base model's state; it is read off the base model on the campaign's own")
print("   training split before the first step -- 05_train's default, no scan.")

# the driver's rule on the live base model over these five frames written as a train file (run.hessian_weight_balance)
import tempfile
from openqha.data import dataset
from openqha_hessian import run as train_run
try:
    with tempfile.TemporaryDirectory() as tmp:
        f = Path(tmp) / "fixture.extxyz"
        dataset._write_split(f, [(a, "train") for a in fr_r], reference=True)
        b = train_run.hessian_weight_balance("MACE-OFF23_medium", f, energy_weight=wE, forces_weight=wF, device="cpu")
    print(f"\nrun.hessian_weight_balance (05_train --hessian-weight balance, the default) on the same five frames, live base model:")
    print(f"  L_E = {b['L_E']:.3e}  L_F = {b['L_F']:.3e}  L_H = {b['L_H']:.3e} (cartesian, {b['N_HESSIAN_FRAMES']} Hessian frames)  ->  w_H = {b['HESSIAN_WEIGHT_BALANCED']:.3f}")
    ratio = b["HESSIAN_WEIGHT_BALANCED"] / (wF * L_F / L_H_cart)
    same = "the same rule, the same number" if abs(ratio - 1) < 0.02 else "DIFFERENT -- the stored fixture values are not from this model"
    print(f"  against the stored-value number above: ratio {ratio:.4f} -> {same}")
    print("  the Record of a run says HESSIAN_WEIGHT_RULE = balance and BALANCE_L_E / L_F / L_H beside HESSIAN_WEIGHT")
except Exception as exc:                                                 # noqa: BLE001
    print("\nbase model not loadable here -- the driver's balance rule skipped:", str(exc)[:160])

base model on the 5 frames:  L_E = 1.635e-07 (eV/atom)²   L_F = 7.223e-04 (eV/Å)²   w_F L_F = 7.223e-02
  L_H (2.2, the target) = 2.450e-02  ->  w_H (4.1) =    2.948
  PHL's w_H/w_F = 0.09/0.30 = 0.30 would put the Cartesian Hessian term at 10.2x the force term here
-> the number depends on the data set and on the base model's state; it is read off the base model on the campaign's own
   training split before the first step -- 05_train's default, no scan (S0-C-60).


/home/ubuntu/anaconda3/envs/openqha/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/openQHA-Hessian/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)



run.hessian_weight_balance (05_train --hessian-weight balance, the default) on the same five frames, live base model:
  L_E = 1.635e-07  L_F = 7.223e-04  L_H = 2.450e-02 (cartesian, 5 Hessian frames)  ->  w_H = 2.948
  against the stored-value number above: ratio 1.0000 -> the same rule, the same number
  the Record of a run says HESSIAN_WEIGHT_RULE = balance and BALANCE_L_E / L_F / L_H beside HESSIAN_WEIGHT


## 5. The training mechanics as the code now does them

The fine-tune is mace's own loop on the mace fork (`BloomDlwlrma/mace`, branch
`openqha-hessian`): `--foundation_model
MACE-OFF23_medium`, `--E0s foundation` (same level, so the energy zero must not move),
`--loss external --loss_module openqha_hessian.phl_loss:build`, float64. Three mechanisms
of mace 0.3.16 decide how the frames are seen; all three were read from the source
and each became a correction or a tool:

**The Replay is its file.** `--multiheads_finetuning True --pt_train_file <file>`
concatenates the file's frames with ours into one shuffled training set: **every frame of
either head once per epoch**, so the replay ratio is a ratio of dataset sizes, not PFT's
$K=4$ step loop. With a user-supplied file `--num_samples_pt` and `--weight_pt_head` are not
read (they act only on mace's own `mp` download path) — so `run.mace_argv` never emits them;
the coverage knob is the file's frame count and the weight knob each frame's `config_weight`,
both written by `scripts/tooling/s0_spice_pt_draw.py` (one seed, one permutation of SPICE's
951,005 train frames, the first $n$ eligible frames in permutation order, so a smaller draw is
a prefix of a larger one;
frames of the forgetting draw's molecules and of the four in_distribution molecules are
skipped; a companion `<stem>.valid.extxyz` of 200 frames from the end of the permutation is the
pretraining head's own validation set — without it mace would take 10 % of the Replay file
for that). Below `--real_pt_data_ratio_threshold` (default 0.1) mace silently duplicates the
fine-tune frames; the driver passes 0. And in multihead mode `run_train.py` used to overwrite
`args.loss = "universal"` — the Hessian term vanished whenever replay was on — until **fork
commit C** guarded it. The Record prints `PT_N_FRAMES`, `PT_CONFIG_WEIGHT`, both heads' counts
from mace's log and `REPLAY_PER_HESSIAN_FRAME = PT_N_FRAMES / N_TRAIN_HESSIAN`; the Dataset
Record (`04_dataset`) prints `N_TRAIN_HESSIAN` and the two production draw commands, and keeps
the S0 ladder's fields (`REPLAY_R4_FRAMES = 4 × N_TRAIN_HESSIAN`,
`dataset.REPLAY_PER_HESSIAN_FRAME_R4`, `REPLAY_CONFIG_WEIGHT_R4 = 10`) as labelled history.

**The S0 rows (history) and the round-1 standard.** The 2026-09-21/22 scan design defined
R0 = 0; R1 = 5,000; R2 = 17,132 (≈ 1 per
Hessian frame on the then draw300); R3 = 68,528 (≈ 4, PFT's $K=4$ as a dataset ratio); R4 = R3's frames
with `config_weight = 10` (PFT's upstream/phonon force-weight ratio). **Round 1's production
standard (the mace-docs multihead guidance; user, 2026-09-30) is instead a 30,000-frame
draw of SPICE's train split consumed by two arms** — `replay30k_w1` / `replay30k_w10`, one seed
so the two frame sets are identical, `config_weight` 1 and 10 (the weight lives in the file) —
submitted together, one job per arm; $w_H$ = the epoch-0 balance. The S0 rows and a $w_H$ scan
are not run until the round-1 numbers are in (`05_train` takes `--hessian-weight <number>` for
such a row; the default is the balance). The cell asserts commit C in the fork's source, that `evaluate` keeps the force graph for the loss (below),
and prints the R0–R4 table (the S0 ladder, history) with the frames per Hessian frame and the **coverage** — the
expected number of distinct SPICE molecules a uniform-by-frame draw of $n$ frames touches,
$\sum_m\big[1-(1-n/N)^{n_m}\big]$ from the SPICE index's per-molecule frame counts.

In [ ]:
import re
try:
    import mace
    fork = Path(mace.__file__).resolve().parents[1]
    src = (fork / "mace" / "cli" / "run_train.py").read_text(encoding="utf-8")
    ev = (fork / "mace" / "tools" / "train.py").read_text(encoding="utf-8")
    guarded = bool(re.search(r'if args\.loss != "external":\s*\n\s*args\.loss = "universal"', src))
    force_graph = 'output_args["force_graph"]' in src and "force_graph" in ev and "eval_summary" in ev
    print(f"mace {mace.__version__} at {fork.name}:")
    print(f"  multihead mode guarded for --loss external (commit C): {'YES' if guarded else 'NO -- the Hessian term is dropped when replay is on'}")
    print(f"  evaluate keeps the force graph and merges the loss's eval_summary (commit C): {'YES' if force_graph else 'NO'}")
    print(f"  --num_samples_pt read outside the mp path: {'yes' if src.count('num_samples_pt') > 1 else 'no (only in the mp/matpes/omat download path)'}")
    assert guarded and force_graph, "the fork at hand is older than commit C"
except ImportError as exc:
    print("fork not importable here -- source check skipped:", str(exc)[:120])

from openqha_hessian import run as train_run
argv = train_run.mace_argv("t.xyz", "v.xyz", "replay30k_w1", "/tmp", "/b.model", LEVEL, multiheads=True,
                           pt_train_file="spice_pt_replay30k_w1.extxyz", pt_valid_file="spice_pt_replay30k_w1.valid.extxyz", max_epochs=100)
pairs = train_run.argv_pairs(argv)
print(f"\nrun.mace_argv with --multiheads: num_samples_pt emitted: {'num_samples_pt' in pairs};  real_pt_data_ratio_threshold = {pairs['real_pt_data_ratio_threshold']};"
      f"  pt_valid_file = {pairs['pt_valid_file']}")

# coverage of a uniform-by-frame draw, from the SPICE index (train file, connectivity level, monomer frames)
from openqha.store import dat
ix = dat.read_tables(ROOT / "data" / "training_sets" / "mace-off23_spice_index.dat")["Index"]
n_m = np.array([int(r["N_FRAMES"]) for r in ix if r["FILE"] == "train" and r["LEVEL"] == "connectivity" and int(r["N_FRAMES"]) > 0], float)
N_src = n_m.sum()
def coverage(n):
    return float(np.sum(1.0 - (1.0 - n / N_src) ** n_m)) if n else 0.0
from openqha.data import dataset
N_H = 17132                                         # the S0-era Hessian-frame count (the basis for R2)
assert dataset.REPLAY_PER_HESSIAN_FRAME_R4 == 4 and dataset.REPLAY_CONFIG_WEIGHT_R4 == 10.0
print(f"the S0 ladder, as the Dataset Record states it (history -- round 1's production Replay is the 30,000-frame draw written twice, weights 1 and 10): REPLAY_R4_FRAMES = {dataset.REPLAY_PER_HESSIAN_FRAME_R4} x N_TRAIN_HESSIAN = {dataset.REPLAY_PER_HESSIAN_FRAME_R4 * N_H} here, config_weight {dataset.REPLAY_CONFIG_WEIGHT_R4:g}")
print(f"\nSPICE train (index): {len(n_m)} molecules with monomer frames, {int(N_src)} monomer frames, {n_m.mean():.1f} per molecule (max {int(n_m.max())});")
print("  the other frames of the 951,005 are dimers, which the draw also takes (their fragments both count as the molecule drawn)")
print("row   frames    config_weight   frames per Hessian frame (N_H = 17,132)   expected distinct molecules   share of the monomer molecules")
for row, n, w in (("R0", 0, 1.0), ("R1", 5000, 1.0), ("R2", 17132, 1.0), ("R3", 68528, 1.0), ("R4", 68528, 10.0)):
    cov = coverage(n)
    print(f"  {row}   {n:6d}        {w:4.1f}              {n / N_H:6.2f}                            {cov:8.0f}                 {cov / len(n_m) * 100:5.1f} %")
print("-> the same file is a different ratio on every Dataset (0.29 / 1 / 4 per Hessian frame here); by-frame drawing weights molecules")
print("   with many conformers (46 monomer frames on average, up to 194), which is why the ids file records frames_of_molecule;")
print("   R1 touches ~29 % of the molecules, R2 ~69 %, R3 ~99 % -- R3 and R4 differ by the pull, not the coverage, at the same cost.")

mace 0.3.16+openqha at openQHA-Hessian:
  multihead mode guarded for --loss external (commit C): YES
  evaluate keeps the force graph and merges the loss's eval_summary (commit C): YES
  --num_samples_pt read outside the mp path: no (only in the mp/matpes/omat download path)

run.mace_argv with --multiheads: num_samples_pt emitted: False;  real_pt_data_ratio_threshold = 0.0;  pt_valid_file = spice_pt_5000.valid.extxyz


R4 as the Dataset Record states it: REPLAY_R4_FRAMES = 4 x N_TRAIN_HESSIAN = 68528 here, config_weight 10

SPICE train (index): 14777 molecules with monomer frames, 682607 monomer frames, 46.2 per molecule (max 194);
  the other frames of the 951,005 are dimers, which the draw also takes (their fragments both count as the molecule drawn)
row   frames    config_weight   frames per Hessian frame (N_H = 17,132)   expected distinct molecules   share of the monomer molecules
  R0        0         1.0                0.00                                   0                   0.0 %
  R1     5000         1.0                0.29                                4249                  28.8 %
  R2    17132         1.0                1.00                               10181                  68.9 %
  R3    68528         1.0                4.00                               14652                  99.2 %
  R4    68528        10.0                4.00                               14652                  99

**Validation on four fixed probes.** The fork's `evaluate` puts the loss
in eval mode and calls the model with the force graph kept (`wants_force_graph_at_eval`);
the Hessian term is then the *same* estimator (2.2) on $k=4$ Rademacher probes whose
generator is seeded from the frame's Label bytes (`phl_loss.FrameConstants.seed`), so the
probes are identical every epoch for a frame and differ between frames: a fixed, unbiased,
reproducible reading at ~4 HVPs per labelled frame against $3N$ for the full matrix
(`wants_hessian_at_eval = False`). It enters the total validation loss that mace's
`ReduceLROnPlateau`, best checkpoint and Stage Two read; the loss's `eval_summary()` joins the
three terms to `results/*.txt` and the Record keeps the three curves per epoch (epoch $-1$ =
the base model). A flat Hessian curve is a warning line ("the term did not act").

**The control is explicit (the base's recipe).** `run.control_settings`: `--lr 0.01
--scheduler_patience 20 --patience 50 --eval_interval 1 --ema`, Stage Two `--swa` at
$\tfrac34$ of the epochs with `--swa_lr = lr/40` (MACE-OFF23: 0.01 → 0.00025) and the Stage
Two weights `--swa_energy_weight 1000 --swa_forces_weight 100` and
$$w_H^{\rm II}=w_H\,\frac{w_F^{\rm II}}{w_F},\tag{5.1}$$
which keeps the Hessian share unchanged when the force weight changes (with mace's defaults
$w_F^{\rm II}=w_F=100$, so $w_H^{\rm II}=w_H$). The fork rebuilds the external loss from the
`swa_*` weights at the switch; every value is in `config.yaml` and the Record's `[Calculation_Info]`.

## 6. The judge table: gate rows and reference rows

The judge reads only shipped paths — `get_hessian` and `hessian_compare` — never
the training estimator, and reports three distributions: **interpolation** (frames of
fine-tuned molecules, held out by frame), **out_of_molecule** (whole molecules never seen),
**in_distribution** (the base's own SPICE molecules, where the question is damage). Its rows
are of two kinds. **Gate rows** decide the verdict: the **Hessian matrix
itself** against the Label on the held-out Hessian frames — the training target's own number
$\|H_\theta-H_r\|_F^2/(9N^2)$, engine no worse than base (`held_out_hessian_cartesian`,
ratio − 1 ≤ 0) — the in_distribution no-degradation (no HIP metric worse than the base by
more than 15 %), the forgetting line (SPICE E/F RMSE ≤ 1.15× the base's). Everything
*computed from the matrix afterwards* is post-processing and a **reference row**: measured
against a number and reported, never gated — the low-mode frequency line (≤ 8.5 cm⁻¹; the
standard vibrational analysis of both matrices, `hessian_compare`), the
msRRHO entropy at the engine's own minima (`MODEL_ERROR_S_REF` ≤ 0.2 cal/mol/K, read from
the msRRHO Records the fine-tuned engine wrote under a tag of its own — branch A and the
`mace` / `compare` steps with `S0_ENGINE=<the name the arm's run printed> … --tag replay30k_w1` (each arm its own tag), the judge's `--thermo-tag replay30k_w1`), the held-out generator's frames binned by RMS displacement (0 / < 0.08
/ < 0.15 / ≥ 0.15 Å) with H, E and F against the base (the NMS reading), and, only when
asked for (`06_judge.py --ramp`), Rodriguez's MD temperature ramp (from each model's own
minimum, +5 K every 5 ps from 5 K until a pair distance's 50-step mean leaves [0.75, 1.5]× its
equilibrium value). **The gate is closed for now**: `06_judge.py` reports every row
against its number and writes `VERDICT = REPORTED`; `--gate` reopens it, and then the
calibration is: the base against itself reads exactly 0 on the Hessian gate and passes; a
potential with the Hessian scaled by 0.81 fails the Hessian gate and with it the verdict. The
cell builds the rows for the five frames from the stored base values, runs `judge.verdict` on
them for the base and for the 0.9-scaled potential, and shows which rows would carry GATE and
what the closed gate answers.

In [ ]:
def hrow(H_eng):
    cs = [hc.compare_hessians(H_eng[k], H_r[k], mm, X[k]) for k in range(nF)]
    return {key: float(np.mean([c[key] for c in cs])) for key in
            ("HESSIAN_MAE", "HESS_REL_FROBENIUS", "FREQ_MAE_CM", "FREQ_MAE_LOW_CM", "SOFTENING_SLOPE", "EIGVEC1_COS_ECKART")}
base = hrow(H_t)
scaled = hrow([0.81 * h for h in H_t])                 # ScaledCalculator(0.9): frequencies x 0.9, Hessian x 0.81
print("out_of_molecule row (2-methyloxirane, 5 frames; in the campaign the four displaced ones are reference bins), H metrics against the wB97M Label:")
print("engine             H MAE     rel.Frob   freq MAE   low-mode MAE   softening   cos v1")
for name, r in (("base", base), ("base scaled 0.9", scaled)):
    print(f"  {name:16s}  {r['HESSIAN_MAE']:.4f}    {r['HESS_REL_FROBENIUS']:.3f}      {r['FREQ_MAE_CM']:5.1f}       {r['FREQ_MAE_LOW_CM']:5.1f}         {r['SOFTENING_SLOPE']:.4f}     {r['EIGVEC1_COS_ECKART']:.3f}")

# the judge's frequency rows average over the frames WITH a Hessian: in the campaign those are the basin frames,
# so the verdict rows below are built from the basin frame alone (the displaced frames' 0.0 low-mode MAE means "no mode below
# 300 cm^-1 at a non-stationary geometry", not a small error)
c0 = hc.compare_hessians(H_t[0], H_r[0], mm, X[0]); s0 = hc.compare_hessians(0.81 * H_t[0], H_r[0], mm, X[0])
cart0 = phl.cartesian_loss_full(H_t[0], H_r[0]); cart_s = phl.cartesian_loss_full(0.81 * H_t[0], H_r[0])
def dist_row(c, cart):
    return [dict(DISTRIBUTION="out_of_molecule", N_FRAMES=1, N_MOLECULES=1, FREQ_MAE_LOW_CM=float(c["FREQ_MAE_LOW_CM"]),
                 FREQ_MAE_CM=float(c["FREQ_MAE_CM"]), HESSIAN_MAE=float(c["HESSIAN_MAE"]), EIGVAL_MAE_ECKART=None,
                 LOSS_CARTESIAN=float(cart), BASE_LOSS_CARTESIAN=float(cart0),
                 BASE_FREQ_MAE_LOW_CM=float(c0["FREQ_MAE_LOW_CM"]), BASE_FREQ_MAE_CM=float(c0["FREQ_MAE_CM"]),
                 BASE_HESSIAN_MAE=float(c0["HESSIAN_MAE"]), BASE_EIGVAL_MAE_ECKART=None)]
ok_forget = dict(N_FRAMES=5000, E_RATIO=1.0, F_RATIO=1.0)     # a forgetting line that passes, so the verdict has a gate row to read
print(f"\njudge.verdict on the basin frame's rows: the target ||dH||^2/(9N^2) base {cart0:.4e}, x0.9 {cart_s:.4e} (ratio {cart_s/cart0:.2f});"
      f" low-mode MAE base {c0['FREQ_MAE_LOW_CM']:.1f}, x0.9 {s0['FREQ_MAE_LOW_CM']:.1f} cm^-1 against 8.5; entropy row '-' (no msRRHO Record here):")
print(f"{'line':36s} {'gate':4s} {'base':>10s} {'x0.9':>10s}")
vb = {l["LINE"]: l for l in judge.verdict(dist_row(c0, cart0), [], ok_forget, noise_floor_cm=24.0)}
vs = {l["LINE"]: l for l in judge.verdict(dist_row(s0, cart_s), [], ok_forget, noise_floor_cm=24.0)}
for line in vb:
    print(f"{line:36s} {vb[line]['GATE']:4s} {vb[line]['RESULT']:>10s} {vs[line]['RESULT']:>10s}")
print(f"{'VERDICT, gate open (--gate)':41s} {judge.verdict_of(list(vb.values()), gate=True):>10s} {judge.verdict_of(list(vs.values()), gate=True):>10s}")
print(f"{'VERDICT, gate closed (default)':41s} {judge.verdict_of(list(vb.values()), gate=False):>10s} {judge.verdict_of(list(vs.values()), gate=False):>10s}")
print("-> the gate reads the matrix: the base against itself is exactly 0 (PASS); the 0.9x potential's ||dH||^2/(9N^2) is many times")
print("   the base's (FAIL, and the verdict with it). The low-mode line -- computed from the matrix -- is a reference row: the base")
print("   reads FAIL on it (12.7 against 8.5, the ring's own error), the 0.9x potential worse (32.1), and neither moves the verdict.")
print("   With the gate closed both read REPORTED: the table is read by eye, and reopening the gate is decided on its numbers.")

out_of_molecule row (2-methyloxirane, 5 frames; in the campaign the four displaced ones are reference bins), H metrics against the wB97M Label:
engine             H MAE     rel.Frob   freq MAE   low-mode MAE   softening   cos v1
  base              0.0769    0.012        5.7         4.1         0.9999     1.000
  base scaled 0.9   0.6078    0.191      155.8        13.7         0.8999     1.000

judge.verdict on the basin frame's rows: the target ||dH||^2/(9N^2) base 2.8686e-02, x0.9 2.2915e+00 (ratio 79.88); low-mode MAE base 12.7, x0.9 32.1 cm^-1 against 8.5; entropy row '-' (no msRRHO Record here):
line                                 gate       base       x0.9
held_out_hessian_cartesian           yes        PASS       FAIL
forgetting                           yes        PASS       PASS
held_out_low_mode_mae_cm             no         FAIL       FAIL
model_error_s_ref_cal_per_mol_K      no            -          -
VERDICT, gate open (--gate)                     PASS       FAIL
VERDICT,

## 7. The recipe, end to end

```bash
# labels: Hessians at the basin frames; E/F at the held-out generators' frames
python workflows/hessian_learning/03_labels.py --tag draw300 --level wb97m-d3bj_def2-tzvppd
# 1. the Dataset: basin frames train and validate; displaced / merged / saddle are held out in test
python workflows/hessian_learning/04_dataset.py --tag draw300 --name draw300 --split-by frame --train-generators basin

#    ... prints N_TRAIN_HESSIAN and the two production draw commands (the S0 ladder fields stay as history)

# 2. the forgetting draw (SPICE test split), then the production Replay -- 30,000 frames of SPICE's train split,
#    one seed, written twice at config_weight 1 and 10 (the same frame set; the weight lives in the file; the
#    mace-docs multihead guidance; user, 2026-09-30)
python scripts/tooling/s0_spice_test_draw.py --n 5000
python scripts/tooling/s0_spice_pt_draw.py --n 30000 --seed 0 --weight 1  --out $S0_RUNS_ROOT/spice/spice_pt_replay30k_w1.extxyz
python scripts/tooling/s0_spice_pt_draw.py --n 30000 --seed 0 --weight 10 --out $S0_RUNS_ROOT/spice/spice_pt_replay30k_w10.extxyz
#    ... spice_pt_replay30k_w{1,10}.extxyz (+ .ids.dat, .toml [Replay]) and the .valid.extxyz companions (200 frames, the pt head's validation)

# 3. the TWO production arms, one A800 each, submitted together: w_H = balance is 05_train's DEFAULT (measured on the
#    base over this train file, Cartesian; HESSIAN_WEIGHT_RULE / BALANCE_L_* in the Record); 4 fixed validation probes,
#    the control explicit; round 1's re-selected knobs ride the same EXTRA line; the Record prints REPLAY_PER_HESSIAN_FRAME
TAG=draw300 RUN=replay30k_w1 MAX_EPOCHS=100 MULTIHEADS=1 \
    EXTRA='--register --register-copy --lr 0.0001 --no-swa --mace-arg=--clip_grad=1.0 --mace-arg=--weight_decay=0.0 --mace-arg=--ema_decay=0.99999' \
    PT_TRAIN_FILE=$S0_RUNS_ROOT/spice/spice_pt_replay30k_w1.extxyz PT_VALID_FILE=$S0_RUNS_ROOT/spice/spice_pt_replay30k_w1.valid.extxyz \
    yhbatch -p ai -G 1 -c 12 -t 24:00:00 hpc/slurm/hl_train.slurm
#    the w10 arm: the same line with RUN=replay30k_w10 and the _w10 paths (nothing else differs between the arms)
#    (the same line by hand:  python workflows/hessian_learning/05_train.py --tag draw300 --name draw300 --run replay30k_w1 --device cuda
#     --max-epochs 100 --multiheads --pt-train-file ... --pt-valid-file ... --register --register-copy --lr 0.0001 --no-swa
#     --mace-arg=--clip_grad=1.0 --mace-arg=--weight_decay=0.0 --mace-arg=--ema_decay=0.99999   -- no --hessian-weight: the balance)
#    the smoke run first (the tianhe test set: one day CREST + one day ORCA): the same five steps with --tag smoke and
#    MAX_EPOCHS=20; its Record's SECONDS_PER_EPOCH sets step 3's walltime (one measurement serves both arms)

# 4. each arm registered itself (--register --register-copy ride its EXTRA; the run prints its ENGINES entry), then its own
#    minima and msRRHO for the pinned seven under a tag of its own (the molecule directory binds branch A's engine,
#    so an arm's Records cannot share the smoke tags)
export S0_ENGINE=<the name the arm's run printed>
python scripts/production/s0_A_pipeline.py --tag replay30k_w1 --species dsgdb9nsd_000035      # x 7
python scripts/production/s0_thermo_msrrho.py --tag replay30k_w1 --species dsgdb9nsd_000035 --step mace
#    (--step reference reuses the ORCA basins of the smoke tags; --step compare writes MODEL_ERROR_S_REF under the arm's tag)

# 5. the judge with the gate CLOSED: every row against its number, VERDICT = REPORTED; the ramp only with --ramp
python workflows/hessian_learning/06_judge.py --tag draw300 --name draw300 --engine <that name> \
    --spice-file data/training_sets/spice_test_5000.extxyz --thermo-tag replay30k_w1
#    what comes next (a w_H row by --hessian-weight <number>, the S0 rows R0-R3, reopening the gate with --gate) is decided on the round-1 numbers
```

What the Record of every row carries: `LOSS phl`, `PROBE gaussian`, `N_PROBES 4`, `HESSIAN_WEIGHT` with
`HESSIAN_WEIGHT_RULE balance` and `BALANCE_L_E / L_F / L_H`, `VALID_PROBES gaussian k=4
fixed`, `PT_N_FRAMES`, `PT_CONFIG_WEIGHT`, `PT_HEAD_TRAIN/VALID`, `FT_HEAD_TRAIN/VALID`,
`REPLAY_PER_HESSIAN_FRAME`, `REAL_PT_DATA_RATIO_THRESHOLD 0`, `LR`, `SCHEDULER_PATIENCE`,
`PATIENCE`, `EMA`, `SWA`, `START_SWA`, `SWA_LR`, both weight sets, `STAGE_TWO_EPOCH`,
`HESSIAN_CURVE_MOVED`, `MACE_FORK_COMMIT` (commit C or later), and the three validation
curves per epoch. Everything beyond the two arms — a `w_H` row, the S0 rows R0–R3, `k ∈ {2, 4}`, `--freeze` — is a
row of the same table only if the round-1 numbers ask for it.

## 8. T05 against T04, and what changed

| | **T04** (the derivation) | **T05** (this: E, F, H to ωB97M at the minima) |
|---|---|---|
| goal | why E/F cannot fix curvature; what the loss estimates; what it costs | one potential: the base model's own level reached on E, F **and** H **at the molecules' minima**; the msRRHO result is the deliverable; extrapolation is measured, not promised |
| level | ωB97M | ωB97M only — the base's level, so `E0s foundation` holds and the base's E/F errors are the yardstick |
| what is trained | fine-tune | same |
| data rule | Proposition 1 at every sample; Taylor (3.1) for what a basin set leaves free | **basin frames only**: displaced / merged / saddle are held-out generator frames; $\rho_k$ and the RMS bins are what the judge *reads* (§3, §6) |
| loss | PHL eq. 6: $\|\Delta H\|_F^2/(3N)^2$, sampled by (2.2); nothing projected but the probe | the same (§2); `phl_loss` as the code runs it |
| probe / HVP | Gaussian (PHL's default) or Rademacher (the smaller variance); $3N$ unit probes $=$ `get_hessian` | the same; `probe gaussian`, `n_probes 4` |
| validation | the full matrix is the deterministic limit of the estimator | **four stored standard-normal probes fixed per frame** in the total validation loss; the full matrix is the judge's tool |
| judge | Algorithm 5 | the table: gate rows (the matrix itself vs base, in_distribution, forgetting) and reference rows computed from the matrix (low modes and the entropy by the standard analysis, RMS bins, MD ramp on request); the gate CLOSED for now; with it open, base-vs-base exactly 0 / 0.81× fails (§6) |
| training mechanics | (5.1), Algorithms 3–4 | §5: the Replay by its file and `config_weight`, threshold 0, commit C, the S0 rows R0–R4 (history) with coverage, the round-1 standard — a 30,000-frame draw in two arms (`replay30k_w1` / `replay30k_w10`) — explicit control, Stage Two by (5.1) |
| $w_H$ | no universal value (T04 §5 item 3) | measured on the base (4.1) by `05_train`'s default `balance`, the three terms in the Record; no scan (§4) |
| acceptance | S1–S7 | A1–A7 below; S1–S7 are inherited, not repeated |

What changed on 2026-09-22 (PHL verbatim): T03's projected, mass-weighted, entropy-weighted
variants of the loss — designed for the msRRHO entropy before the target moved to the
matrix — are no longer carried as "diagnostics" in these notebooks (they were removed from the
code); the band table that compared where each norm spends its gradient is gone with them; the
frequency rows of the judge are the standard vibrational analysis of the trained matrix, stated as
such. Everything else — basin-only training with the held-out generator as the extrapolation
readout, the fixed-probe validation, the Replay as a drawn file,
the gate on the matrix itself and its closure, the production chain — stands.

**Acceptance for this goal.**

| id | statement | cell | result here |
|---|---|---|---|
| A1 | the base model's E, F, H errors against its own level are measurable on one fixture, on all three quantities at once | §1 | table |
| A2 | the training term (2.2) is the Cartesian Frobenius norm sampled and unbiased; the code's $k=4$ draw has the predicted s.e.; the $3N$ unit probes are exact | §2 | Hutchinson to noise; s.e. $=$ T04 eq. 2.3; exact to every digit |
| A3 | the fine-tune trains on the basin row only; the displaced rows are held-out generator frames whose reference-curvature change (15–22 %) and $\rho_k$ the judge reads as reference bins | §3 | table with the reader of each row |
| A4 | $w_H$ from (4.1) is a property of the data set and the base model; PHL's ratio does not transfer; the driver's `balance` rule on the live base reproduces the stored-value number | §4 | numbers + `run.hessian_weight_balance` |
| A5 | the fork's multihead path keeps `--loss external` and `evaluate` keeps the force graph (commit C, asserted); the driver emits no `num_samples_pt`, threshold 0, a `pt_valid_file`; the S0 rows R0–R4 with the per-Hessian-frame ratio and the coverage per row (the ladder's record fields asserted); round 1's production Replay is the 30,000-frame draw in two arms — the file IS the size | §5 | assertion + table |
| A6 | the judge gates on the Hessian matrix itself (the training target vs base), in_distribution and forgetting; the low-mode line, the entropy, the RMS bins and the ramp are computed from the matrix and are reference rows; the 0.81× Hessian fails the gate and the verdict | §6 | `judge.verdict` table with GATE |
| A7 | the recipe is the README's five steps: the Dataset, the two production draws (30,000 frames, one seed, `config_weight` 1 / 10), the two arms (`replay30k_w1` / `replay30k_w10`; $w_H$ = the balance by default), registration and each engine's own msRRHO under the arm's tag, the judge with the gate closed and the arm's `--thermo-tag`; the smoke run first | §7 | listing |

References: PHL — *Projected Hessian Learning (PHL): Fast Curvature Supervision for Accurate
MLIPs* (eq. 6, Algorithm 1; reference implementation `PHL-main/PHL_training.ipynb`); Rodriguez, Smith,
Mendoza-Cortes, *JCTC* 2025, 10.1021/acs.jctc.5c00402; Koker et al., *PFT: Phonon Fine-tuning*,
arXiv:2601.07742; Burger et al., *HIP: Hessian Interatomic Potentials Without Derivatives* (§5: the
frequency analysis after mass-weighting and Eckart projection); Kovács et al., *MACE-OFF*, *JACS* 147,
17598 (2025), 10.1021/jacs.4c07099, `mace-off-main/train_scripts`; Hutchinson, *Commun. Stat. Simul.*
18, 1059 (1989); T04 of this folder.